<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>

# **Space X  Falcon 9 First Stage Landing Prediction**
 ## Lab 2: Data wrangling 

En este laboratorio, realizaremos un Análisis Exploratorio de Datos (EDA) para encontrar algunos patrones en los datos y determinar cuál sería la etiqueta para entrenar modelos supervisados.

En el conjunto de datos, hay varios casos diferentes en los que el propulsor no aterrizó con éxito. A veces se intentó un aterrizaje pero falló debido a un accidente; por ejemplo, <code>True Ocean</code> significa que el resultado de la misión fue aterrizar con éxito en una región específica del océano, mientras que <code>False Ocean</code> significa que el resultado de la misión fue aterrizar sin éxito en una región específica del océano. <code>True RTLS</code> significa que el resultado de la misión fue aterrizar con éxito en una plataforma terrestre, y <code>False RTLS</code> significa que el resultado de la misión fue aterrizar sin éxito en una plataforma terrestre. <code>True ASDS</code> significa que el resultado de la misión fue aterrizar con éxito en un buque drone, y <code>False ASDS</code> significa que el resultado de la misión fue aterrizar sin éxito en un buque drone. <br>
En este laboratorio principalmente vamos a convertir esos resultados en etiquetas de entrenamiento, donde `1` significa que el acelerador aterrizó con éxito y `0` significa que no tuvo éxito.

## Objectives
Perform exploratory  Data Analysis and determine Training Labels 

- Exploratory Data Analysis
- Determine Training Labels 
## Import Libraries and Define Auxiliary Functions


In [1]:
import pandas as pd
import numpy as np

### Data Analysis 
Load Space X dataset, from last section.

In [2]:
df=pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv")
df.head(10)

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857
5,6,2014-01-06,Falcon 9,3325.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1005,-80.577366,28.561857
6,7,2014-04-18,Falcon 9,2296.000000,ISS,CCAFS SLC 40,True Ocean,1,False,False,True,NaN,1.0,0,B1006,-80.577366,28.561857
7,8,2014-07-14,Falcon 9,1316.000000,LEO,CCAFS SLC 40,True Ocean,1,False,False,True,NaN,1.0,0,B1007,-80.577366,28.561857
8,9,2014-08-05,Falcon 9,4535.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1008,-80.577366,28.561857
9,10,2014-09-07,Falcon 9,4428.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1011,-80.577366,28.561857


In [3]:
# Identify and calculate the percentage of the missing values in each attribute
df.isnull().sum()/len(df)*100

FlightNumber       0.000000
Date               0.000000
BoosterVersion     0.000000
PayloadMass        0.000000
Orbit              0.000000
LaunchSite         0.000000
Outcome            0.000000
Flights            0.000000
GridFins           0.000000
Reused             0.000000
Legs               0.000000
LandingPad        28.888889
Block              0.000000
ReusedCount        0.000000
Serial             0.000000
Longitude          0.000000
Latitude           0.000000
dtype: float64

In [4]:
df.dtypes

FlightNumber        int64
Date                  str
BoosterVersion        str
PayloadMass       float64
Orbit                 str
LaunchSite            str
Outcome               str
Flights             int64
GridFins             bool
Reused               bool
Legs                 bool
LandingPad            str
Block             float64
ReusedCount         int64
Serial                str
Longitude         float64
Latitude          float64
dtype: object

### TAREA 1: Calcula el número de lanzamientos en cada sitio

Los datos contienen varias instalaciones de lanzamiento de Space X: <a href='https://es.wikipedia.org/wiki/Lista_de_instalaciones_de_lanzamiento_de_Cape_Canaveral_y_Merritt_Island'>Complejo de Lanzamiento 40 de Cape Canaveral Space</a> <b>VAFB SLC 4E</b>, Base de la Fuerza Aérea de Vandenberg Complejo de Lanzamiento 4E <b>(SLC-4E)</b>, Centro Espacial Kennedy Complejo de Lanzamiento 39A <b>KSC LC 39A</b>. La ubicación de cada lanzamiento se coloca en la columna <code>LaunchSite</code>
A continuación, veamos el número de lanzamientos para cada sitio. <br>
Usa el método <code>value_counts()</code> en la columna <code>LaunchSite</code> para determinar el número de lanzamientos en cada sitio:

In [5]:
#  Apply value_counts() on column LaunchSite
df['LaunchSite'].value_counts()

LaunchSite
CCAFS SLC 40    55
KSC LC 39A      22
VAFB SLC 4E     13
Name: count, dtype: int64

Each launch aims to an dedicated orbit, and here are some common orbit types:

* <b>LEO</b>: La órbita terrestre baja (LEO, por sus siglas en inglés) es una órbita centrada en la Tierra con una altitud de 2,000 km (1,200 mi) o menos (aproximadamente un tercio del radio de la Tierra),[1] o con al menos 11.25 periodos por día (un periodo orbital de 128 minutos o menos) y una excentricidad menor a 0.25.[2] La mayoría de los objetos artificiales en el espacio exterior están en LEO <a href='https://en.wikipedia.org/wiki/Low_Earth_orbit'>[1]</a>. 
* <b>VLEO</b>: Las órbitas terrestres muy bajas (VLEO, por sus siglas en inglés) se pueden definir como las órbitas con una altitud media por debajo de los 450 km. Operar en estas órbitas puede proporcionar varios beneficios a las naves espaciales de observación de la Tierra, ya que la nave opera más cerca del objeto de observación<a href='https://www.researchgate.net/publication/271499606_Very_Low_Earth_Orbit_mission_concepts_for_Earth_Observation_Benefits_and_challenges'>[2]</a>.

* <b>GTO</b>(Órbita de Transferencia Geostacionaria): Una órbita de transferencia geostacionaria es una órbita elíptica de la Tierra utilizada para trasladar satélites desde una órbita baja terrestre (LEO) hasta una órbita geostacionaria (GEO). En una GTO, el perigeo (punto más cercano a la Tierra) está mucho más bajo que la altitud GEO, mientras que el apogeo (punto más lejano) alcanza aproximadamente 22,236 millas (35,786 kilómetros) sobre el ecuador de la Tierra, la altitud de una órbita geostacionaria. Los satélites en GTO utilizan propulsión a bordo para circularizar su órbita a la altitud GEO, donde pueden ofrecer servicios como monitoreo del clima, comunicaciones y vigilancia. <a href="https://www.space.com/29222-geosynchronous-orbit.html" >[3] </a>.

* <b>SSO (o SO)</b>: Es una órbita heliosincrónica también llamada órbita solar-síncrona, y es una órbita casi polar alrededor de un planeta, en la que el satélite pasa sobre cualquier punto determinado de la superficie del planeta a la misma hora solar media local <a href="https://en.wikipedia.org/wiki/Sun-synchronous_orbit">[4] <a>.
* <b>ES-L1</b>: En los puntos de Lagrange, las fuerzas gravitacionales de los dos cuerpos grandes se cancelan de tal manera que un objeto pequeño colocado en órbita allí está en equilibrio con respecto al centro de masa de los cuerpos grandes. L1 es uno de esos puntos entre el sol y la Tierra <a href="https://en.wikipedia.org/wiki/Lagrange_point#L1_point">[5]</a>.

* <b>HEO</b>: Una órbita altamente elíptica, es una órbita elíptica con alta excentricidad, generalmente refiriéndose a una alrededor de la Tierra <a href="https://en.wikipedia.org/wiki/Highly_elliptical_orbit">[6]</a>.

* <b>ISS</b>: Una estación espacial modular (satélite artificial habitable) en órbita terrestre baja. Es un proyecto colaborativo multinacional entre cinco agencias espaciales participantes: NASA (Estados Unidos), Roscosmos (Rusia), JAXA (Japón), ESA (Europa) y CSA (Canadá) <a href="https://en.wikipedia.org/wiki/International_Space_Station">[7]</a>

* <b> MEO </b> Órbitas geocéntricas que varían en altitud desde 2.000 km (1.200 mi) hasta justo por debajo de la órbita geoestacionaria a 35.786 kilómetros (22.236 mi). También se conoce como órbita circular intermedia. Estas suelen estar a 20.200 kilómetros (12.600 mi) o 20.650 kilómetros (12.830 mi), con un período orbital de 12 horas <a href="https://en.wikipedia.org/wiki/List_of_orbits"> [8] </a>

 * <b> HEO </b> Órbitas geocéntricas por encima de la altitud de la órbita geoestacionaria (35.786 km o 22.236 mi) <a href="https://en.wikipedia.org/wiki/List_of_orbits"> [9] </a> 
 * <b> GEO </b> Es una órbita geoestacionaria circular a 35.786 kilómetros (22.236 millas) sobre el ecuador de la Tierra y siguiendo la dirección de rotación de la Tierra <a href="https://en.wikipedia.org/wiki/Geostationary_orbit"> [10] </a>


* <b>PO</b> Es un tipo de satélites en el que un satélite pasa por encima o casi por encima de ambos polos del cuerpo que está orbitando (usualmente un planeta como la Tierra <a href="https://en.wikipedia.org/wiki/Polar_orbit"> [11] </a>

algunos se muestran en la siguiente gráfica:

![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/Orbits.png)

### TASK 2: Calculate the number and occurrence of each orbit
 Use the method  <code>.value_counts()</code> to determine the number and occurrence of each orbit in the  column <code>Orbit</code>

Note: Do not count GTO, as it is a transfer orbit and not itself geostationary.


In [6]:
# Apply value_counts on Orbit column
df['Orbit'].value_counts()

Orbit
GTO      27
ISS      21
VLEO     14
PO        9
LEO       7
SSO       5
MEO       3
ES-L1     1
HEO       1
SO        1
GEO       1
Name: count, dtype: int64

### TASK 3: Calculate the number and occurence of mission outcome of the orbits
Usa el método <code>.value_counts()</code> en la columna <code>Outcome</code> para determinar el número de <code>landing_outcomes</code>. Luego asígnalo a una variable llamada landing_outcomes.

In [7]:
# landing_outcomes = values on Outcome column
df['Outcome'].value_counts()

Outcome
True ASDS      41
None None      19
True RTLS      14
False ASDS      6
True Ocean      5
False Ocean     2
None ASDS       2
False RTLS      1
Name: count, dtype: int64

<code>True Ocean</code> significa que el resultado de la misión aterrizó con éxito en una región específica del océano, mientras que <code>False Ocean</code> significa que el resultado de la misión no logró aterrizar en una región específica del océano. <code>True RTLS</code> significa que el resultado de la misión aterrizó con éxito en una plataforma terrestre, y <code>False RTLS</code> significa que no logró aterrizar en una plataforma terrestre. <code>True ASDS</code> significa que el resultado de la misión aterrizó con éxito en un barco dron, y <code>False ASDS</code> significa que no logró aterrizar en un barco dron. <code>None ASDS</code> y <code>None None</code> representan un fallo al aterrizar.

In [8]:
landing_outcomes = df['Outcome'].value_counts()
for i,outcome in enumerate(landing_outcomes.keys()):
    print(i,outcome)

0 True ASDS
1 None None
2 True RTLS
3 False ASDS
4 True Ocean
5 False Ocean
6 None ASDS
7 False RTLS


In [9]:
# We create a set of outcomes where the second stage did not land successfully:
bad_outcomes = set(landing_outcomes.keys()[[1,3,5,6,7]])
bad_outcomes

{'False ASDS', 'False Ocean', 'False RTLS', 'None ASDS', 'None None'}

### TASK 4: Create a landing outcome label from Outcome column
Usando <code>Outcome</code>, crea una lista donde el elemento sea cero si la fila correspondiente en <code>Outcome</code> está en el conjunto <code>bad_outcome</code>; de lo contrario, será uno. Luego asígnala a la variable <code>landing_class</code>:

In [10]:
# Using the Outcome, create a list where the element is zero if the corresponding row in Outcome is in the set bad_outcome; otherwise, it's one. Then assign it to the variable landing_class:
landing_class = df['Outcome'].apply(lambda x: 0 
                                    if x in bad_outcomes 
                                    else 
                                    1)

Esta variable representará la variable de clasificación que indica el resultado de cada lanzamiento. Si el valor es cero, la primera etapa no aterrizó con éxito; uno significa que la primera etapa aterrizó con éxito

In [11]:
df['Class'] = landing_class
df[['Class']].head(8)

,Class
0,0
1,0
2,0
3,0
4,0
5,0
6,1
7,1


Podemos usar la siguiente línea de código para determinar la tasa de éxito:

In [12]:
df['Class'].mean()

np.float64(0.6666666666666666)

We can now export it to a CSV for the next section,but to make the answers consistent, in the next lab we will provide data in a pre-selected date range.

In [13]:
df.to_csv("dataset_part_2.csv", index=False)

In [18]:
# Nombres únicos de los sitios de lanzamiento
print(df['LaunchSite'].unique())

# Equivalente a GROUP BY LaunchSite en MySQL
(df.groupby('LaunchSite')
   .size()
   .reset_index(name='total_lanzamientos')
   .sort_values('total_lanzamientos', ascending=False))

<StringArray>
['CCAFS SLC 40', 'VAFB SLC 4E', 'KSC LC 39A']
Length: 3, dtype: str


,LaunchSite,total_lanzamientos
0,CCAFS SLC 40,55
1,KSC LC 39A,22
2,VAFB SLC 4E,13
